In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Training Pipeline - Validador\n",
    "Genera embeddings, entrena múltiples modelos, guarda predicciones y compara PTM/IPTM real vs predicho."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "import torch\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.metrics import mean_squared_error, r2_score\n",
    "\n",
    "from models.esm2_embedder import ESM2Embedder\n",
    "from models.transformer_encoder import TransformerEncoder\n",
    "from models.mlp_predictor import MLPPredictor\n",
    "from models.random_forest_model import RandomForestModel\n",
    "from models.xgboost_model import XGBoostModel\n",
    "from models.ridge_model import RidgeModel\n",
    "from models.mlp_sklearn_model import MLPBaselineModel"

    # Detectar GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Device:", device)

    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))


   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1️⃣ Cargar dataset procesado"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "df = pd.read_csv('../data/processed_dataset.csv')\n",
    "print(df.shape)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2️⃣ Generar embeddings ESM2"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "embedder = ESM2Embedder(device=device, batch_size=8)  # batch_size=1 para secuencias multicadena\n",
    "\n",
    "sequences = df['seq'].tolist()\n",
    "embeddings = embedder.embed(sequences)\n",
    "print('Embeddings shape:', embeddings.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3️⃣ Preparar datos para modelos clásicos y deep learning"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "X = embeddings.cpu().numpy()\n",
    "y = df[['PTM','IPTM']].values\n",
    "print('X shape:', X.shape, 'y shape:', y.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4️⃣ Entrenar ESM2 + Transformer + MLP"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "import torch.nn as nn\n",
    "\n",
    "embeddings_dl = embeddings.unsqueeze(1)  # batch_first=True\n",
    "\n",
    "transformer = TransformerEncoder(embed_dim=embeddings.shape[1]).to(device)\n",
    "mlp = MLPPredictor(input_dim=embeddings.shape[1]).to(device)\n",
    "\n",
    "optimizer = torch.optim.Adam(list(transformer.parameters())+list(mlp.parameters()), lr=1e-4)\n",
    "loss_fn = nn.MSELoss()\n",
    "\n",
    "epochs = 10\n",
    "\n",
    "for epoch in range(epochs):\n",
    "    optimizer.zero_grad()\n",
    "    encoded = transformer(embeddings_dl)\n",
    "    pooled = encoded.mean(1)\n",
    "    preds = mlp(pooled)\n",
    "    loss = loss_fn(preds, torch.tensor(y,dtype=torch.float32).to(device))\n",
    "    loss.backward()\n",
    "    optimizer.step()\n",
    "    print(f'Epoch {epoch+1}/{epochs} - Loss: {loss.item():.5f}')\n",
    "\n",
    "# Guardar modelo\n",
    "torch.save({'transformer': transformer.state_dict(), 'mlp': mlp.state_dict()}, 'outputs/esm2_transformer_mlp.pt')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5️⃣ Entrenar modelos clásicos (sklearn) para comparación"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "rf = RandomForestModel()\n",
    "rf.train(X,y)\n",
    "\n",
    "xgb = XGBoostModel()\n",
    "xgb.train(X,y)\n",
    "\n",
    "ridge = RidgeModel()\n",
    "ridge.train(X,y)\n",
    "\n",
    "mlp_skl = MLPBaselineModel()\n",
    "mlp_skl.train(X,y)\n",
    "\n",
    "# Guardar modelos\n",
    "rf.save('outputs/rf_model.pkl')\n",
    "xgb.save('outputs/xgb_model.pkl')\n",
    "ridge.save('outputs/ridge_model.pkl')\n",
    "mlp_skl.save('outputs/mlp_skl_model.pkl')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6️⃣ Generar predicciones de todos los modelos"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "predictions = {}\n",
    "\n",
    "# ESM2 + Transformer + MLP\n",
    "with torch.no_grad():\n",
    "    encoded = transformer(embeddings_dl)\n",
    "    pooled = encoded.mean(1)\n",
    "    preds_dl = mlp(pooled).cpu().numpy()\n",
    "predictions['ESM2_Transformer_MLP'] = preds_dl\n",
    "\n",
    "# Sklearn models\n",
    "predictions['RandomForest'] = rf.predict(X)\n",
    "predictions['XGBoost'] = xgb.predict(X)\n",
    "predictions['Ridge'] = ridge.predict(X)\n",
    "predictions['MLP_sklearn'] = mlp_skl.predict(X)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7️⃣ Comparar PTM/IPTM real vs predicho y calcular métricas"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "for model_name,preds in predictions.items():\n",
    "    print(f'=== {model_name} ===')\n",
    "    mse_ptm = mean_squared_error(y[:,0], preds[:,0])\n",
    "    mse_iptm = mean_squared_error(y[:,1], preds[:,1])\n",
    "    r2_ptm = r2_score(y[:,0], preds[:,0])\n",
    "    r2_iptm = r2_score(y[:,1], preds[:,1])\n",
    "    print(f'PTM MSE: {mse_ptm:.5f}, R2: {r2_ptm:.5f}')\n",
    "    print(f'IPTM MSE: {mse_iptm:.5f}, R2: {r2_iptm:.5f}')\n",
    "    \n",
    "    # graficas\n",
    "    plt.figure(figsize=(5,4))\n",
    "    plt.scatter(y[:,0], preds[:,0], alpha=0.5)\n",
    "    plt.xlabel('Real PTM')\n",
    "    plt.ylabel('Predicted PTM')\n",
    "    plt.title(f'{model_name} PTM')\n",
    "    plt.show()\n",
    "    \n",
    "    plt.figure(figsize=(5,4))\n",
    "    plt.scatter(y[:,1], preds[:,1], alpha=0.5)\n",
    "    plt.xlabel('Real IPTM')\n",
    "    plt.ylabel('Predicted IPTM')\n",
    "    plt.title(f'{model_name} IPTM')\n",
    "    plt.show()\n",
    "\n",
    "# Guardar todas las predicciones en un CSV\n",
    "df_preds = df[['seq','PTM','IPTM']].copy()\n",
    "for model_name, preds in predictions.items():\n",
    "    df_preds[f'PTM_pred_{model_name}'] = preds[:,0]\n",
    "    df_preds[f'IPTM_pred_{model_name}'] = preds[:,1]\n",
    "\n",
    "df_preds.to_csv('outputs/predictions_all_models.csv', index=False)\n",
    "print('Predicciones guardadas en outputs/predictions_all_models.csv')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# Training Pipeline - Validador\n',
    'Genera embeddings, entrena múltiples modelos, guarda predicciones y compara PTM/IPTM real vs predicho.']},
  {'cell_type': 'code',
   'metadata': {},
   'source': ['import torch\n',
    'import pandas as pd\n',
    'import numpy as np\n',
    'import matplotlib.pyplot as plt\n',
    'from sklearn.metrics import mean_squared_error, r2_score\n',
    '\n',
    'from models.esm2_embedder import ESM2Embedder\n',
    'from models.transformer_encoder import TransformerEncoder\n',
    'from models.mlp_predictor import MLPPredictor\n',
    'from models.random_forest_model import RandomForestModel\n',
    'from models.xgboost_model import XGBoostModel\n',
    'from models.ridge_model import RidgeModel\n',
    'from models.mlp_sklearn_model import MLPBaselineModel']},
  {'cell_type': 'markdown',
   'metadata': {},
   'source': ['## 1️⃣ Cargar dataset procesado']},
  {'cell_type': 'code'

: 